In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score
import sys
sys.path.append('..')
from utils.extract_features import extract_features

print("Импорты выполнены.")

Импорты выполнены.


In [2]:
def tpr_at_1pct_fpr(y_true, y_pred_proba):
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    sorted_true = y_true[sorted_indices]
    total_neg = np.sum(y_true == 0)
    if total_neg == 0:
        return 0.0
    target_fp = int(0.01 * total_neg)
    fp = 0
    tp = 0
    for label in sorted_true:
        if label == 0:
            fp += 1
        else:
            tp += 1
        if fp >= target_fp:
            break
    total_pos = np.sum(y_true == 1)
    if total_pos == 0:
        return 0.0
    return tp / total_pos

In [3]:
df = pd.read_csv("../data/processed/balanced_unified_dataset.csv")  # путь к общему файлу
print(f"Всего строк: {len(df)}")
print("Источники:", df['source'].unique())

Всего строк: 70032
Источники: ['fwaf' 'csic2010' 'vulnxss' 'kaggle' 'httpparams' 'payloadbox']


In [4]:
print("Извлечение признаков...")
features_list = []
for i, text in enumerate(df['text']):
    if i % 1000 == 0:
        print(f"Обработано {i}/{len(df)}")
    features_list.append(extract_features(text, False))

X_all = pd.DataFrame(features_list)
y_all = df['label'].values
sources = df['source'].values
print(f"Форма признаков: {X_all.shape}")

Извлечение признаков...
Обработано 0/70032
Обработано 1000/70032
Обработано 2000/70032
Обработано 3000/70032
Обработано 4000/70032
Обработано 5000/70032
Обработано 6000/70032
Обработано 7000/70032
Обработано 8000/70032
Обработано 9000/70032
Обработано 10000/70032
Обработано 11000/70032
Обработано 12000/70032
Обработано 13000/70032
Обработано 14000/70032
Обработано 15000/70032
Обработано 16000/70032
Обработано 17000/70032
Обработано 18000/70032
Обработано 19000/70032
Обработано 20000/70032
Обработано 21000/70032
Обработано 22000/70032
Обработано 23000/70032
Обработано 24000/70032
Обработано 25000/70032
Обработано 26000/70032
Обработано 27000/70032
Обработано 28000/70032
Обработано 29000/70032
Обработано 30000/70032
Обработано 31000/70032
Обработано 32000/70032
Обработано 33000/70032
Обработано 34000/70032
Обработано 35000/70032
Обработано 36000/70032
Обработано 37000/70032
Обработано 38000/70032
Обработано 39000/70032
Обработано 40000/70032
Обработано 41000/70032
Обработано 42000/70032


In [5]:
source_names = ['fwaf', 'csic2010', 'kaggle', 'vulnxss', 'httpparams']
results = {}

for test_source in source_names:
    print(f"\n===== Тестируем на {test_source} =====")
    
    test_mask = (sources == test_source)
    train_mask = ~test_mask
    
    X_train = X_all[train_mask].values
    X_test = X_all[test_mask].values
    y_train = y_all[train_mask]
    y_test = y_all[test_mask]
    
    print(f"  Обучающих: {len(X_train)}, тестовых: {len(X_test)}")
    
    # Масштабирование
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Создание и обучение SVM
    # Параметры: ядро RBF, C=1.0, gamma='scale' — стандартные
    # probability=True нужно для вероятностей
    model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42, class_weight='balanced')
    model.fit(X_train_scaled, y_train)
    
    # Предсказания
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    
    # Метрики
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    tpr = tpr_at_1pct_fpr(y_test, y_proba)
    
    results[test_source] = {
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'f1': f1,
        'accuracy': acc,
        'tpr@1%fpr': tpr
    }
    
    print(f"  ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}, F1: {f1:.4f}, Accuracy: {acc:.4f}, TPR@1%FPR: {tpr:.4f}")

print("\nLODO для SVM завершён.")


===== Тестируем на fwaf =====
  Обучающих: 35907, тестовых: 34125


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 0.0000, F1: 0.0000, Accuracy: 0.8111, TPR@1%FPR: 0.0000

===== Тестируем на csic2010 =====
  Обучающих: 44873, тестовых: 25159


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.5787, Accuracy: 0.4071, TPR@1%FPR: 0.0000

===== Тестируем на kaggle =====
  Обучающих: 62689, тестовых: 7343
  ROC-AUC: 0.7864, PR-AUC: 0.9961, F1: 0.9830, Accuracy: 0.9666, TPR@1%FPR: 0.0001

===== Тестируем на vulnxss =====
  Обучающих: 68504, тестовых: 1528


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.9627, Accuracy: 0.9280, TPR@1%FPR: 0.0000

===== Тестируем на httpparams =====
  Обучающих: 69005, тестовых: 1027
  ROC-AUC: 0.8552, PR-AUC: 0.4535, F1: 0.6796, Accuracy: 0.7907, TPR@1%FPR: 0.0000

LODO для SVM завершён.


In [6]:
result_df = pd.DataFrame(results).T
print("\n=== Итоговая таблица LODO (SVM RBF) ===")
print(result_df.round(4))

result_df.to_csv("lodo_svm.csv")
print("Результаты сохранены в lodo_svm.csv")


=== Итоговая таблица LODO (SVM RBF) ===
            roc_auc  pr_auc      f1  accuracy  tpr@1%fpr
fwaf            NaN  0.0000  0.0000    0.8111     0.0000
csic2010        NaN  1.0000  0.5787    0.4071     0.0000
kaggle       0.7864  0.9961  0.9830    0.9666     0.0001
vulnxss         NaN  1.0000  0.9627    0.9280     0.0000
httpparams   0.8552  0.4535  0.6796    0.7907     0.0000
Результаты сохранены в lodo_svm.csv
